# Notebook 02 — Pré-processamento de Texto

**Objetivo:** Limpar e normalizar os textos do dataset para prepará-los para a vetorização e o treinamento dos modelos.

O pré-processamento é uma etapa crítica em NLP (Processamento de Linguagem Natural). Textos brutos contêm ruído (tags HTML, pontuação, variações de capitalização) que prejudicam a qualidade dos modelos. Cada etapa abaixo tem uma justificativa técnica.

**Pipeline de pré-processamento aplicado:**
1. Remoção de tags HTML
2. Tratamento de apóstrofos
3. Remoção de caracteres especiais e números
4. Conversão para minúsculas
5. Tokenização
6. Remoção de stopwords + Stemming

In [2]:
# ===== SETUP: montar o Drive e carregar o dataset bruto =====
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/tcc-sentiment-analysis/'

# Carregamos o dataset BRUTO aqui (não o já processado)
# pois este notebook é responsável por gerar o imdb_clean.csv
df = pd.read_csv(DRIVE_PATH + 'data/IMDB Dataset.csv')
print(f"✅ Dataset carregado: {df.shape}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Dataset carregado: (50000, 2)


## 1. Importações e Download de Recursos NLTK

Utilizamos a biblioteca **NLTK** (Natural Language Toolkit) para:
- `stopwords`: lista de palavras muito frequentes em inglês que pouco contribuem para o sentido (ex: "the", "is", "and")
- `punkt`: tokenizador de sentenças
- `PorterStemmer`: algoritmo de stemming que reduz palavras à sua raiz (ex: "running" → "run")

In [3]:
import nltk
import re

# Baixar recursos necessários do NLTK (executar apenas uma vez)
nltk.download('stopwords')
nltk.download('punkt')

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


## 2. Definição da Função de Pré-processamento

A função `preprocess()` aplica o pipeline completo de limpeza a um único texto.

**Decisões técnicas importantes:**
- **Apóstrofos → espaço** (antes de remover especiais): evita que "you'll" vire "youll" — sem essa etapa, a contração ficaria como uma única palavra desconhecida
- **Stemming com PorterStemmer**: algoritmo baseado em regras, não em dicionário. Produz raízes artificiais ("episode" → "episod") que são consistentes ao longo do dataset — modelos aprendem que são a mesma raiz
- **Stopwords em inglês**: lista padrão do NLTK com ~179 palavras muito frequentes removidas

In [4]:
# Inicializar ferramentas de NLP
stop_words = set(stopwords.words('english'))  # conjunto para busca O(1)
stemmer = PorterStemmer()

def preprocess(text):
    """Aplica o pipeline completo de limpeza a um texto.

    Args:
        text (str): Review bruto do IMDB

    Returns:
        str: Texto limpo e normalizado
    """
    # Etapa 1: Remover tags HTML (ex: <br />, <b>, </b>)
    text = re.sub(r'<.*?>', '', text)

    # Etapa 2: Substituir apóstrofos por espaço
    # Evita que contrações como "you'll" virem "youll" após remoção de especiais
    text = re.sub(r"'", ' ', text)

    # Etapa 3: Remover caracteres especiais e números
    # Mantém apenas letras e espaços
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Etapa 4: Converter para minúsculas
    # "Movie" e "movie" devem ser tratados como a mesma palavra
    text = text.lower()

    # Etapa 5: Tokenizar (separar em lista de palavras)
    tokens = text.split()

    # Etapa 6: Remover stopwords e aplicar stemming
    # Filtramos stopwords e reduzimos cada palavra à sua raiz
    tokens = [stemmer.stem(t) for t in tokens if t not in stop_words]

    return ' '.join(tokens)

## 3. Verificação: Antes e Depois do Pré-processamento

É fundamental verificar visualmente o resultado da função antes de aplicá-la ao dataset completo. O texto processado deve:
- Não conter tags HTML
- Estar todo em minúsculas
- Não conter stopwords ("the", "is", "and", etc.)
- Conter palavras no formato stemmed (raiz artificial)

In [5]:
# Verificar o resultado do pré-processamento com um exemplo real
exemplo = df['review'][0]

print("ANTES (primeiros 300 caracteres):")
print(exemplo[:300])

print("\nDEPOIS:")
print(preprocess(exemplo[:300]))

ANTES (primeiros 300 caracteres):
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Tru

DEPOIS:
one review mention watch oz episod hook right exactli happen meth first thing struck oz brutal unflinch scene violenc set right word go tru


## 4. Aplicação ao Dataset Completo

Aplicamos a função `preprocess()` a todas as 50.000 reviews usando `pandas.apply()`.

> **Tempo estimado:** ~2 minutos no Google Colab (processamento sequencial de 50k textos)

In [6]:
# Aplicar o pré-processamento a toda a coluna 'review'
# O resultado é armazenado em uma nova coluna 'review_clean'
print("Aplicando pré-processamento... (pode demorar ~2 minutos)")
df['review_clean'] = df['review'].apply(preprocess)
print("Concluído!")

# Exibir comparação lado a lado das primeiras 3 linhas
df[['review', 'review_clean', 'sentiment']].head(3)

Aplicando pré-processamento... (pode demorar ~2 minutos)
Concluído!


,review,review_clean,sentiment
0,One of the other reviewers has mentioned that ...,one review mention watch oz episod hook right ...,positive
1,A wonderful little production. <br /><br />The...,wonder littl product film techniqu unassum old...,positive
2,I thought this was a wonderful way to spend ti...,thought wonder way spend time hot summer weeke...,positive


## 5. Validação do Dataset Processado

Antes de salvar, verificamos se o processamento gerou algum valor nulo ou string vazia. Isso pode acontecer se um review contiver apenas stopwords ou caracteres especiais após a limpeza.

In [7]:
# Verificar integridade do dataset após processamento
vazios = df['review_clean'].isna().sum()
vazios_str = (df['review_clean'] == '').sum()

print(f"Nulos: {vazios}")          # Esperado: 0
print(f"Strings vazias: {vazios_str}")  # Esperado: 0
print(f"Total de reviews: {len(df)}")

print(f"\nExemplo aleatório (review processado):")
print(df['review_clean'].sample(1).values[0][:200])

Nulos: 0
Strings vazias: 0
Total de reviews: 50000

Exemplo aleatório (review processado):
dare say film better origin good right comedi film good origin though mani scene get laugh think themth stori film even bizarr origin make great peter hewitt great job direct film great cast core cast


## 6. Salvamento do Dataset Processado

Salvamos o resultado como `imdb_clean.csv` no Google Drive. Este arquivo será usado pelos notebooks seguintes (03 e 04) como entrada para o treinamento dos modelos.

In [8]:
# Salvar o dataset processado no Google Drive
# Este arquivo é a entrada para o notebook 03_models
df.to_csv(DRIVE_PATH + 'data/imdb_clean.csv', index=False)
print("✅ Dataset salvo com sucesso em: data/imdb_clean.csv")

✅ Dataset salvo com sucesso em: data/imdb_clean.csv


---
## Conclusões do Pré-processamento

- Pipeline de 6 etapas aplicado com sucesso a todas as **50.000 reviews**
- **0 valores nulos** e **0 strings vazias** após o processamento
- Dataset salvo como `imdb_clean.csv` pronto para vetorização
- O vocabulário foi reduzido significativamente com a remoção de stopwords e stemming

**Próximo passo:** `03_models.ipynb`